In [1]:
import pandas as pd
import numpy as np
import pickle

from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import root_mean_squared_error , mean_absolute_error , mean_absolute_percentage_error          

pd.options.display.float_format = '{:,.2f}'.format
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

## 1. Data Preparation

In [2]:
path = '../datasets/'

df = pd.read_csv(f"{path}/dataset_preprocessed.csv")


In [7]:
df

,date,month,workforce_diff,inactives_diff,employment_rate_diff,activity_rate_diff,industry_production_prices_diff,companies_trust,gasoline_price_diff,diesel_price_diff,GPL_price,diesel_heatining_price_diff,unemployment_rate,unemployment_rate_diff,rolling_mean_6m,rolling_mean_12m,PCA_labour_market_1,PCA_labour_market_2,PCA_price_energy_1,PCA_price_energy_2
0,2006-04-01,4,"64,766.00","-64,702.00",0.44,0.16,0.50,104.10,0.05,0.02,"66,035.00",0.01,6.94,-0.45,-0.28,-0.08,0.39,0.02,0.87,0.08
1,2006-05-01,5,"73,541.00","-73,878.00",0.18,0.20,0.20,103.00,0.04,0.03,"65,159.00",0.02,6.93,-0.01,-0.22,-0.04,0.47,0.01,1.00,0.04
2,2006-06-01,6,"-174,782.00","175,114.00",0.03,-0.46,0.20,103.40,-0.00,-0.01,"65,159.00",-0.01,6.21,-0.72,-0.26,-0.12,-1.21,0.02,-0.25,0.04
3,2006-07-01,7,"-176,787.00","178,782.00",-0.14,-0.46,0.80,100.80,0.03,0.01,"65,159.00",0.01,5.74,-0.47,-0.36,-0.07,-1.23,0.02,0.57,0.05
4,2006-08-01,8,"-280,442.00","281,478.00",-0.79,-0.74,0.00,100.70,0.00,0.00,"65,113.00",0.00,5.90,0.16,-0.36,-0.05,-1.93,0.02,0.05,0.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,2025-09-01,9,"214,183.00","-223,229.00",0.08,0.60,0.30,93.80,0.01,0.00,"69,714.00",0.01,6.04,0.72,-0.13,0.03,1.46,-0.02,0.15,0.20
234,2025-10-01,10,"-19,881.00","16,427.00",0.23,-0.05,-0.30,94.30,-0.02,-0.01,"69,545.00",-0.01,5.64,-0.40,-0.10,-0.02,-0.16,0.01,-0.47,0.19
235,2025-11-01,11,"-286,624.00","285,359.00",-0.65,-0.77,1.20,96.00,0.02,0.06,"69,413.00",0.03,5.52,-0.12,-0.26,-0.03,-1.98,0.02,1.25,0.15
236,2025-12-01,12,"105,547.00","-108,877.00",0.07,0.29,-0.90,96.50,-0.02,-0.03,"69,482.00",-0.06,5.83,0.31,-0.16,-0.08,0.69,0.00,-0.82,0.20


In [8]:
df_input_forecast = pd.read_csv(f"{path}/dataset_input_forecast.csv")

In [9]:
df_input_forecast

,month,employment_rate_diff_fcst,industry_production_prices_diff_fcst,companies_trust_fcst,rolling_mean_6m_fcst,rolling_mean_12m_fcst,PCA_labour_market_1_fcst,PCA_labour_market_2_fcst,PCA_price_energy_1_fcst,PCA_price_energy_2_fcst
0,2,-0.08,-0.39,96.55,-0.10,-0.12,-0.10,-0.02,0.08,0.61
1,3,-0.02,-0.15,96.33,-0.15,-0.12,0.15,0.01,-0.04,-0.19
2,4,0.03,0.71,96.12,-0.05,-0.12,-0.09,0.01,-0.20,0.03
3,5,0.17,1.09,96.04,-0.05,-0.19,1.03,0.00,0.00,-0.11
4,6,-0.09,0.10,95.86,-0.06,-0.15,-1.56,-0.00,0.09,-0.12
5,7,-0.14,0.25,95.64,0.09,-0.11,-0.71,0.01,0.27,-0.21
6,8,0.03,0.16,95.36,0.07,-0.11,0.44,-0.01,-0.37,-0.18
7,9,0.08,0.29,95.25,-0.00,-0.14,1.25,0.01,0.07,-0.21
8,10,0.09,0.45,94.86,-0.03,-0.07,-0.51,0.00,0.02,0.16
9,11,-0.13,-0.14,94.26,-0.02,-0.07,-0.63,0.00,-0.02,-0.01


In [3]:
def data_preparation(df):
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')

    X_columns = df.drop(['unemployment_rate','unemployment_rate_diff'], axis = 1)

    selected_cols = ['month', 
                'employment_rate_diff', 
                'industry_production_prices_diff',
                'companies_trust',  
                'rolling_mean_6m',
                'rolling_mean_12m', 
                'PCA_labour_market_1',
                'PCA_labour_market_2', 
                'PCA_price_energy_1', 
                'PCA_price_energy_2'
                ]

    exog_train = X_columns.iloc[5:-12][selected_cols]
    exog_test = X_columns.iloc[-12:][selected_cols]

    y_train = df.iloc[5:-12]['unemployment_rate_diff']
    y_test = df.iloc[-12:]['unemployment_rate_diff']

    return exog_train, exog_test, y_train, y_test


In [22]:
# exog_train, exog_test, y_train, y_test = data_preparation(df)

## 2. Model: SARIMAX

In [23]:
# # best_param = (1, 1, 0)
# # best_param_seasonal = (0, 1, 0, 12)

# best_param = (0, 0, 0)
# best_param_seasonal = (1, 0, 0, 12)

In [4]:
def model(df):
    exog_train, exog_test, y_train, y_test = data_preparation(df)
    
    best_param = (0, 0, 0)
    best_param_seasonal = (1, 0, 0, 12)
    
    model = SARIMAX(y_train,
                    exog=exog_train, 
                    order=best_param,
                    seasonal_order=best_param_seasonal,
                    enforce_stationarity=False,
                    enforce_invertibility=False)
    results = model.fit()

    y_test_pred = results.predict(
                            start=exog_test.index[0],
                            end=exog_test.index[-1],
                            exog=exog_test
                                )

    test_rmse = round(root_mean_squared_error(y_test, y_test_pred), 5 )
    print(f"RMSE: {test_rmse}")

    anchor =  df['unemployment_rate'].iloc[-13] 

    df_prection = pd.DataFrame({
        "date":y_test_pred.index,
        "unemployment_rate_fcst":anchor + y_test_pred.cumsum().values
        })

    df_final = df.merge(df_prection, on= 'date', how = 'left')
    
    return  df_final , results   

In [7]:
df_final, results = model(df)

RMSE: 0.02812


In [9]:
selected_cols = ['month', 
                'employment_rate_diff', 
                'industry_production_prices_diff',
                'companies_trust',  
                'rolling_mean_6m',
                'rolling_mean_12m', 
                'PCA_labour_market_1',
                'PCA_labour_market_2', 
                'PCA_price_energy_1', 
                'PCA_price_energy_2'
                ]

with open('../models/sarimax_unemployment_12m.pkl', 'wb') as f:
    pickle.dump({'model': results, 'selected_cols': selected_cols}, f)

In [ ]:
df_final.to_csv(f"{path}dataset_predicion_training.csv")